# Figure 4 · Same galaxies, same scale, every reconstruction

**Question:** Which observed structures and overall shapes are recovered from NISP?

**Status:** waiting for completed runs and a frozen prediction export. This notebook
does not run models. It will show NISP, JWST, legacy, direct-5×, diffusion and
the diffusion mean image, with matched physical display scales and residuals.

Select examples using input brightness/SNR, size and crowding before inspecting
model outputs. Include a faint or difficult example; do not select only successes.
The initial layout uses the first four IDs in the approved bundle.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Works from the repository root or from paper_figures/.
here = Path.cwd()
workspace = here if (here / 'figure_utils.py').exists() else here / 'paper_figures'
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))
from figure_utils import (ROOT, FIGURES, DATA, COLORS, style, read_json, export,
                          panel, box, arrow, verified_sources, completed_evaluation,
                          run_inventory, sha)
%matplotlib inline
style()
VERSION = 'v01'

In [2]:
N_EXAMPLES = 4
PREDICTIONS = ['legacy', 'direct5', 'diffusion', 'diffusion_mean3']
bundle_path = FIGURES / 'inputs/galaxy_examples.npz'
metadata_path = FIGURES / 'inputs/galaxy_examples.json'
ready = bundle_path.exists() and metadata_path.exists()
if not ready:
    display(Markdown('**Waiting for the frozen prediction bundle.** See `inputs/README.md`. No inference is launched.'))
else:
    metadata = read_json(metadata_path)
    assert metadata['complete'] and metadata['split'] == 'validation'
    assert metadata['units'] == 'MJy/sr' and metadata['npz_sha256'] == sha(bundle_path)
    with np.load(bundle_path, allow_pickle=False) as saved:
        images = dict(saved)
    ids = images['object_ids']
    assert len(np.unique(ids)) == len(ids)
    np.testing.assert_array_equal(ids, metadata['object_ids'])
    assert images['nisp'].shape == images['nisp_valid'].shape == (len(ids), 41, 41)
    assert images['jwst'].shape == images['jwst_valid'].shape == (len(ids), 205, 205)
    for name in PREDICTIONS:
        assert images[name].shape == images['jwst'].shape and metadata['checkpoint_sha256'][name]
    print('Approved object IDs:', ids[:N_EXAMPLES].tolist())

**Waiting for the frozen prediction bundle.** See `inputs/README.md`. No inference is launched.

In [3]:
# All stretches are display-only; measurements must use the original linear arrays.
if ready:
    count = min(N_EXAMPLES, len(ids))
    names = ['nisp', 'jwst', *PREDICTIONS]
    titles = ['NISP Y', 'JWST F115W', 'Residual legacy', 'Direct 5×', 'Diffusion sample', 'Diffusion mean (3)']
    fig, axes = plt.subplots(2*count, len(names), figsize=(10.5, 2.1*count),
                             gridspec_kw={'height_ratios':[1, .55]*count})
    cmap = plt.get_cmap('gray').copy(); cmap.set_bad('#888888')
    residual_cmap = plt.get_cmap('RdBu_r').copy(); residual_cmap.set_bad('#888888')
    for i in range(count):
        valid = images['jwst_valid'][i].astype(bool)
        values = images['jwst'][i][valid]
        low, high = np.percentile(values, [.5, 99.5])
        low, high = min(0., low), max(high, low+1e-8)
        stretch = max((high-low)*.03, 1e-6)
        residual_limit = max((high-low)*.3, stretch)
        scale = metadata['nisp_pixel_scale_arcsec'][i]
        extent = [-20.5*scale, 20.5*scale, -20.5*scale, 20.5*scale]
        for j, name in enumerate(names):
            mask = images['nisp_valid'][i].astype(bool) if name == 'nisp' else valid
            arr = np.ma.array(images[name][i], mask=~mask)
            axes[2*i,j].imshow(np.arcsinh(arr/stretch), origin='lower', extent=extent, interpolation='nearest',
                cmap=cmap, vmin=np.arcsinh(low/stretch), vmax=np.arcsinh(high/stretch))
            if i == 0: axes[2*i,j].set_title(titles[j], fontsize=8)
            if j == 0: axes[2*i,j].set_ylabel(f'ID {ids[i]}', fontsize=8)
            if name in PREDICTIONS:
                residual = np.ma.array(images[name][i]-images['jwst'][i], mask=~valid)
                axes[2*i+1,j].imshow(residual, origin='lower', extent=extent, interpolation='nearest',
                    cmap=residual_cmap, vmin=-residual_limit, vmax=residual_limit)
            else:
                axes[2*i+1,j].axis('off')
        axes[2*i,0].plot([extent[0]+.6, extent[0]+2.6], [extent[2]+.6]*2, color='white', lw=2)
        axes[2*i,0].text(extent[0]+1.6, extent[2]+.9, '2 arcsec', color='white', ha='center', fontsize=7)
        axes[2*i+1,1].text(.5, .5, f'SR − JWST\n±{residual_limit:.3g} MJy/sr',
                          transform=axes[2*i+1,1].transAxes, ha='center', va='center', fontsize=7)
    for ax in axes.flat: ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle('Validation examples · shared intensity scales within each galaxy', fontsize=11)
    fig.tight_layout()
    plt.show()

In [4]:
if ready:
    paths = export(fig, f'fig04_galaxy_examples_{VERSION}', '04_galaxy_examples.ipynb',
        [bundle_path, metadata_path], dict(object_ids=ids[:count].tolist(), predictions=PREDICTIONS,
        scale='shared asinh intensity; symmetric linear residuals'), status='validation example draft')

## Working caption and review choices

Matched validation galaxies shown in native NISP sampling, observed JWST and
the model reconstructions. Display limits are shared within each galaxy;
residuals use a symmetric scale and invalid pixels remain masked. Intensities
are sky-subtracted MJy/sr. The different Y/F115W bands and PSFs remain part of
the prediction problem.

- Final row count, crop size and the input-only example-selection rule remain
  open for discussion; record actual IDs before exporting model predictions.
- Consider central crops plus one full-field context row.
- A single diffusion sample and the three-sample mean must be labeled separately.
- The 2-arcsec bar uses stored per-object pixel area, not an assumed nominal scale.
- Do not select a different checkpoint or different galaxy for each method.